

# Feature Matching (특징점 매칭)

## 개요

Feature Matching은 서로 다른 이미지에서 추출된 특징점들 사이의 대응 관계를 찾는 과정입니다. COLMAP은 **SIFT descriptor** 기반의 매칭을 사용하며, **ratio test**와 **cross-check**를 통해 신뢰도 높은 매칭만 선별합니다.

---

## 이론적 배경

### 1. Descriptor Distance

두 SIFT descriptor $\mathbf{d}_1$과 $\mathbf{d}_2$ 사이의 거리:

#### L2 Distance (Euclidean)
$$
d_{L2}(\mathbf{d}_1, \mathbf{d}_2) = \|\mathbf{d}_1 - \mathbf{d}_2\|_2 = \sqrt{\sum_{i=1}^{128} (d_{1i} - d_{2i})^2}
$$

#### Dot Product (for normalized descriptors)
$$
\text{similarity} = \mathbf{d}_1 \cdot \mathbf{d}_2 = \sum_{i=1}^{128} d_{1i} \cdot d_{2i}
$$

정규화된 descriptor의 경우:
$$
d_{L2}^2 = 2(1 - \mathbf{d}_1 \cdot \mathbf{d}_2)
$$

따라서 dot product가 클수록 L2 거리가 작음.

#### COLMAP의 Distance Type 선택

| 매칭 방식 | Distance Type | 사용 조건 |
|-----------|---------------|-----------|
| **Brute-Force** | Dot Product | GPU 또는 CPU (`cpu_brute_force=true`) |
| **Index-based** (FLANN/FAISS) | L2 Distance | CPU (`cpu_brute_force=false`) |

> **Note**: Distance type은 GPU/CPU의 차이가 아니라 **Brute-Force vs Index-based**의 차이입니다.
> - **Brute-Force**: 모든 쌍을 행렬곱 한 번으로 계산 → Dot Product가 효율적
> - **Index-based**: KD-tree, IVF 등 L2 metric 기반으로 설계된 자료구조 사용

3DGS의 `convert.py`는 `--SiftMatching.use_gpu 1`을 사용하므로 **GPU Brute-Force + Dot Product** 방식입니다.

### 2. Nearest Neighbor Matching

이미지 1의 각 특징점 $\mathbf{f}_i^{(1)}$에 대해, 이미지 2에서 가장 가까운 특징점을 찾음:

$$
\text{match}(\mathbf{f}_i^{(1)}) = \arg\min_{\mathbf{f}_j^{(2)}} d(\mathbf{d}_i^{(1)}, \mathbf{d}_j^{(2)})
$$

### 3. Lowe's Ratio Test

**문제**: 단순 nearest neighbor는 많은 false positive 발생

**해결**: 첫 번째 best match와 두 번째 best match의 거리 비율 확인

$$
\frac{d_{\text{best}}}{d_{\text{second best}}} < \tau_{\text{ratio}}
$$

일반적으로 $\tau_{\text{ratio}} = 0.8$ (COLMAP 기본값)

**직관적 의미**:
- 비율이 낮음 → best match가 확실히 더 좋음 (distinctive)
- 비율이 높음 → 두 후보가 비슷함 (ambiguous)

### 4. Cross-Check (Mutual Nearest Neighbor)

**양방향 일관성 확인**:

$$
\text{valid match}(\mathbf{f}_i^{(1)}, \mathbf{f}_j^{(2)}) \iff 
\begin{cases}
\text{NN}(\mathbf{f}_i^{(1)}) = \mathbf{f}_j^{(2)} \\
\text{NN}(\mathbf{f}_j^{(2)}) = \mathbf{f}_i^{(1)}
\end{cases}
$$

이미지 1 → 이미지 2 매칭과 이미지 2 → 이미지 1 매칭이 일치해야 유효

### 5. Distance Threshold

절대적인 거리 threshold로 매칭 품질 보장:

$$
d(\mathbf{d}_1, \mathbf{d}_2) < \tau_{\text{distance}}
$$

SIFT descriptor는 [0, 255] 범위이므로, normalized distance 사용:
$$
d_{\text{normalized}} = \arccos\left(\frac{\mathbf{d}_1 \cdot \mathbf{d}_2}{\|\mathbf{d}_1\| \|\mathbf{d}_2\|}\right)
$$

---

## Matching 전략

### Exhaustive Matching ✅ (3DGS 기본)
모든 이미지 쌍에 대해 매칭 수행
- 시간 복잡도: $O(N^2)$ (N = 이미지 수)
- 소규모~중규모 데이터셋에 적합 (수백 장 이하)

### Sequential Matching (비디오용)
연속된 이미지들 사이에서만 매칭
- 비디오 프레임에 적합
- 시간 복잡도: $O(N)$

---

## COLMAP 구현 분석

### Matching Options

In [ ]:
// sift.h - SIFT 매칭 옵션

struct SiftMatchingOptions {
  // Ratio test threshold
  // 첫 번째와 두 번째 best match 거리 비율의 최대값
  double max_ratio = 0.8;

  // 절대 거리 threshold (normalized)
  // descriptor 거리가 이 값보다 크면 매칭 거부
  double max_distance = 0.7;

  // Cross-check 활성화 여부
  // true: 양방향 일관성 확인 (더 정확, 더 느림)
  bool cross_check = true;

  // Brute-force vs Index-based 매칭
  // brute-force: 모든 쌍 계산 (정확하지만 느림)
  // index-based: FAISS 등 사용 (빠름)
  bool cpu_brute_force_matcher = false;

  // Descriptor index 캐시 (재사용으로 속도 향상)
  ThreadSafeLRUCache<image_t, FeatureDescriptorIndex>*
      cpu_descriptor_index_cache = nullptr;
};



### Matching Options (General)

In [ ]:
// matcher.h - 일반 매칭 옵션

struct FeatureMatchingOptions {
  // 스레드 수 (-1: 자동)
  int num_threads = -1;

  // GPU 사용 여부
  bool use_gpu = true;  // CUDA 지원 시

  // GPU 인덱스 (멀티 GPU: "0,1,2,3")
  std::string gpu_index = "-1";

  // 이미지 쌍당 최대 매칭 수
  int max_num_matches = 32768;

  // Guided matching (기하학적 제약 사용)
  bool guided_matching = false;

  // Geometric verification 건너뛰기
  bool skip_geometric_verification = false;
};



### Brute-Force Matching (One-Way)

In [ ]:
// sift.cc - 한 방향 brute-force 매칭

size_t FindBestMatchesOneWayBruteForce(
    const Eigen::RowMajorMatrixXf& dot_products,  // [N1 x N2] dot product matrix
    const float max_ratio,                         // ratio test threshold
    const float max_distance,                      // distance threshold
    std::vector<int>* matches) {                   // output: matches[i] = j
  
  // Normalized SIFT descriptor의 squared norm은 512*512
  constexpr float kInvSqDescriptorNorm = 1.0f / (512 * 512);

  size_t num_matches = 0;
  matches->resize(dot_products.rows(), -1);  // -1: no match

  for (Eigen::Index i1 = 0; i1 < dot_products.rows(); ++i1) {
    int best_d2_idx = -1;
    float best_dot_product = 0;
    float second_best_dot_product = 0;

    // 이미지 2의 모든 descriptor와 비교
    for (Eigen::Index i2 = 0; i2 < dot_products.cols(); ++i2) {
      const float dot_product = dot_products(i1, i2);
      
      if (dot_product > best_dot_product) {
        // 새로운 best
        best_d2_idx = i2;
        second_best_dot_product = best_dot_product;
        best_dot_product = dot_product;
      } else if (dot_product > second_best_dot_product) {
        // 새로운 second best
        second_best_dot_product = dot_product;
      }
    }

    // 매칭을 찾지 못함
    if (best_d2_idx == -1) continue;

    // Dot product → L2 distance 변환 (normalized descriptor)
    // d² = 2(1 - dot/norm²) → d = arccos(dot/norm²)
    const float best_dist_normed =
        std::acos(std::min(kInvSqDescriptorNorm * best_dot_product, 1.0f));

    // Distance threshold 체크
    if (best_dist_normed > max_distance) continue;

    const float second_best_dist_normed = std::acos(
        std::min(kInvSqDescriptorNorm * second_best_dot_product, 1.0f));

    // Ratio test: best >= ratio * second_best 이면 reject
    // (ambiguous match 제거)
    if (best_dist_normed >= max_ratio * second_best_dist_normed) continue;

    ++num_matches;
    (*matches)[i1] = best_d2_idx;
  }

  return num_matches;
}



**수식과 코드의 대응:**

| 이론 | 코드 |
|------|------|
| $d = \arccos(\mathbf{d}_1 \cdot \mathbf{d}_2 / \|\mathbf{d}\|^2)$ | `std::acos(kInvSqDescriptorNorm * dot_product)` |
| $d_{\text{best}} < \tau_{\text{dist}}$ | `best_dist_normed > max_distance` |
| $d_{\text{best}} / d_{\text{second}} < \tau_{\text{ratio}}$ | `best_dist >= max_ratio * second_best_dist` |

### Cross-Check 구현

In [ ]:
void FindBestMatchesBruteForce(
    const Eigen::RowMajorMatrixXf& dot_products,
    const float max_ratio,
    const float max_distance,
    const bool cross_check,
    FeatureMatches* matches) {
  
  matches->clear();

  // 1→2 방향 매칭
  std::vector<int> matches_1to2;
  const size_t num_matches_1to2 = FindBestMatchesOneWayBruteForce(
      dot_products, max_ratio, max_distance, &matches_1to2);

  if (cross_check) {
    // 2→1 방향 매칭 (transpose 사용)
    std::vector<int> matches_2to1;
    const size_t num_matches_2to1 = FindBestMatchesOneWayBruteForce(
        dot_products.transpose(), max_ratio, max_distance, &matches_2to1);
    
    matches->reserve(std::min(num_matches_1to2, num_matches_2to1));
    
    // Cross-check: 양방향 일치 확인
    for (size_t i1 = 0; i1 < matches_1to2.size(); ++i1) {
      if (matches_1to2[i1] != -1 &&                         // i1 → j 매칭 존재
          matches_2to1[matches_1to2[i1]] != -1 &&          // j → ? 매칭 존재
          matches_2to1[matches_1to2[i1]] == static_cast<int>(i1)) {  // j → i1
        // Mutual nearest neighbor!
        FeatureMatch match;
        match.point2D_idx1 = i1;
        match.point2D_idx2 = matches_1to2[i1];
        matches->push_back(match);
      }
    }
  } else {
    // Cross-check 없음: 1→2 매칭만 사용
    matches->reserve(num_matches_1to2);
    for (size_t i1 = 0; i1 < matches_1to2.size(); ++i1) {
      if (matches_1to2[i1] != -1) {
        FeatureMatch match;
        match.point2D_idx1 = i1;
        match.point2D_idx2 = matches_1to2[i1];
        matches->push_back(match);
      }
    }
  }
}



### CPU Matcher 클래스

In [ ]:
// sift.cc - CPU 기반 매처

class SiftCPUFeatureMatcher : public FeatureMatcher {
 public:
  explicit SiftCPUFeatureMatcher(const FeatureMatchingOptions& options)
      : options_(options) {
    THROW_CHECK(options_.Check());
  }

  void Match(const Image& image1,
             const Image& image2,
             FeatureMatches* matches) override {
    THROW_CHECK_NOTNULL(matches);
    THROW_CHECK_NOTNULL(image1.descriptors);
    THROW_CHECK_NOTNULL(image2.descriptors);
    THROW_CHECK_EQ(image1.descriptors->cols(), 128);  // SIFT 128-D
    THROW_CHECK_EQ(image2.descriptors->cols(), 128);

    matches->clear();

    if (image1.descriptors->rows() == 0 || 
        image2.descriptors->rows() == 0) {
      return;  // 특징점 없음
    }

    // Descriptor index 캐싱 (재사용)
    if (!options_.sift->cpu_brute_force_matcher) {
      if (prev_image_id1_ != image1.image_id) {
        index1_ = options_.sift->cpu_descriptor_index_cache->Get(image1.image_id);
        prev_image_id1_ = image1.image_id;
      }
      if (prev_image_id2_ != image2.image_id) {
        index2_ = options_.sift->cpu_descriptor_index_cache->Get(image2.image_id);
        prev_image_id2_ = image2.image_id;
      }
    }

    if (options_.sift->cpu_brute_force_matcher) {
      // Brute-force: dot product matrix 계산
      const Eigen::RowMajorMatrixXf dot_products =
          ComputeSiftDistanceMatrix(DistanceType::DOT_PRODUCT,
                                    nullptr, nullptr,
                                    *image1.descriptors,
                                    *image2.descriptors,
                                    nullptr);
      FindBestMatchesBruteForce(dot_products,
                                options_.sift->max_ratio,
                                options_.sift->max_distance,
                                options_.sift->cross_check,
                                matches);
    } else {
      // Index-based: KNN 검색
      Eigen::RowMajorMatrixXi indices_1to2;
      Eigen::RowMajorMatrixXf l2_dists_1to2;
      
      // image2의 index에서 image1의 descriptor 검색
      index2_->Search(/*num_neighbors=*/2,
                      image1.descriptors->cast<float>(),
                      indices_1to2,
                      l2_dists_1to2);
      
      Eigen::RowMajorMatrixXi indices_2to1;
      Eigen::RowMajorMatrixXf l2_dists_2to1;
      
      if (options_.sift->cross_check) {
        index1_->Search(/*num_neighbors=*/2,
                        image2.descriptors->cast<float>(),
                        indices_2to1,
                        l2_dists_2to1);
      }

      FindBestMatchesIndex(indices_1to2, l2_dists_1to2,
                           indices_2to1, l2_dists_2to1,
                           options_.sift->max_ratio,
                           options_.sift->max_distance,
                           options_.sift->cross_check,
                           matches);
    }
  }

 private:
  const FeatureMatchingOptions options_;
  image_t prev_image_id1_ = kInvalidImageId;
  image_t prev_image_id2_ = kInvalidImageId;
  std::shared_ptr<FeatureDescriptorIndex> index1_;
  std::shared_ptr<FeatureDescriptorIndex> index2_;
};



> **Note:** Guided Matching은 기하학적 제약(Epipolar/Homography)을 이용한 고급 기법입니다.
> 3DGS의 `convert.py`에서는 기본적으로 **비활성화** 상태입니다.

---

## 이론과 코드 매핑

| 이론적 개념 | COLMAP 구현 |
|------------|-------------|
| L2 distance | `ComputeSiftDistanceMatrix(DistanceType::L2)` |
| Dot product | `ComputeSiftDistanceMatrix(DistanceType::DOT_PRODUCT)` |
| Ratio test | `best_dist >= max_ratio * second_best_dist` |
| Cross-check | `matches_2to1[matches_1to2[i1]] == i1` |
| Distance threshold | `best_dist > max_distance` |
| Epipolar constraint | `guided_filter` with Sampson error |
| Homography constraint | `guided_filter` with transfer error |

---

## 실용적 고려사항

### 1. 파라미터 튜닝

In [ ]:
// 더 많은 매칭을 원할 때 (recall ↑, precision ↓)
options.max_ratio = 0.9;       // 기본 0.8 → 더 관대
options.max_distance = 0.8;    // 기본 0.7 → 더 관대
options.cross_check = false;   // 비활성화 (약간 더 많은 매칭)

// 더 정확한 매칭을 원할 때 (recall ↓, precision ↑)
options.max_ratio = 0.7;       // 더 엄격한 ratio test
options.max_distance = 0.6;    // 더 엄격한 거리 threshold
options.cross_check = true;    // 양방향 확인



### 2. CPU vs GPU

| 구분 | CPU (FAISS) | GPU (SiftGPU) |
|-----|------------|---------------|
| 속도 | 빠름 | 매우 빠름 |
| 메모리 | 낮음 | GPU 메모리 필요 |
| 정확도 | 동일 | 동일 |
| 배치 | 어려움 | 쉬움 |

### 3. Matching 전략 비교

| 전략 | 복잡도 | 적합한 상황 | 3DGS |
|-----|-------|-----------|:----:|
| Exhaustive | $O(N^2)$ | 소규모~중규모 (수백 장) | ✅ |
| Sequential | $O(N)$ | 비디오 시퀀스 | ✅ |

### 4. Geometric Verification

매칭 후 반드시 **geometric verification** 수행:

1. **RANSAC + F/E/H 추정**: Outlier 제거
2. **Inlier count threshold**: 최소 매칭 수 확인
3. **Inlier ratio threshold**: 매칭 품질 확인

```
Raw matches (descriptor 기반)
        ↓
Geometric verification (RANSAC)
        ↓
Inlier matches (기하학적으로 일관된 매칭만)
```



---

## 3DGS (convert.py) 기본 설정

3DGS의 `convert.py`는 다음 COLMAP 매칭을 사용합니다:

In [ ]:
# Exhaustive matching (기본)
colmap exhaustive_matcher \
    --database_path $DATABASE \
    --SiftMatching.use_gpu 1

# 또는 Sequential matching (--skip_matching 없이 비디오용)
colmap sequential_matcher \
    --database_path $DATABASE \
    --SiftMatching.use_gpu 1



### 사용되는 기법

| 기법 | 사용 여부 | COLMAP 기본값 | 목적 |
|------|:--------:|:-------------:|------|
| Ratio Test | ✅ | 0.8 | Ambiguous match 제거 |
| Cross-Check | ✅ | true | 양방향 일관성 |
| Distance Threshold | ✅ | 0.7 | 품질 보장 |
| GPU 가속 | ✅ | true | 속도 향상 |
| Geometric Verification | ✅ | true | Outlier 제거 |
| Guided Matching | ❌ | false | 속도 우선 |

### 파이프라인 흐름

```
convert.py
    │
    ▼
colmap exhaustive_matcher
    │
    ├── 1. Descriptor Distance (Dot Product)
    │       모든 이미지 쌍에 대해 SIFT descriptor 비교
    │
    ├── 2. Ratio Test (0.8)
    │       d_best / d_second_best < 0.8
    │
    ├── 3. Distance Threshold (0.7)
    │       d_best < 0.7 (normalized)
    │
    ├── 4. Cross-Check
    │       A→B 매칭과 B→A 매칭이 일치해야 유효
    │
    └── 5. Geometric Verification (RANSAC)
            F/E/H matrix 추정하며 outlier 제거
            → database.db에 저장
```



---

## 참고 문헌

1. Lowe, D.G. "Distinctive Image Features from Scale-Invariant Keypoints" (IJCV 2004)
2. Muja, Lowe. "Fast Approximate Nearest Neighbors with Automatic Algorithm Configuration" (VISAPP 2009)
3. Fischler, Bolles. "Random Sample Consensus" (1981) - RANSAC
4. Johnson, Douze. "Billion-scale similarity search with GPUs" (2017) - FAISS